
# 14_v13 Sensitivity analysis using saved final 14_v7 predictions

This notebook **does not refit the models**.

Reason:
- the strict first-occurrence-like definition changes only which observed outbreak events are included in the sensitivity evaluation;
- the final `14_v7` model for each event week was trained using only observations before that week, independently of the later event-classification label;
- the saved final `14_v7_all_grid_predictions_event_weeks.parquet` contains predictions for all 5,491 national grid cells in all 35 event weeks.

Therefore, the safest sensitivity analysis is to:
1. recompute event classification under alternative look-back/radius definitions;
2. join those events to the **already validated final 14_v7 all-grid predictions**;
3. recompute Top-k capture, percentile, gain/loss, paired bootstrap CI, and exact discordant-transition test.

This avoids any risk of changing the final model training or sampling procedure.


**v12 correction:** week-coverage validation is applied only to weeks that contain events classified as strict first-occurrence-like under at least one sensitivity definition, rather than to all 61 evaluation-event weeks.


**v13 correction:** the paired-bootstrap random-number generator is reset for each sensitivity definition. Thus, definitions yielding the same event set also yield the same bootstrap confidence interval, and the primary 8w/30km definition reproduces the final 14_v7 bootstrap CI.


In [ ]:

# 1. Setup and robust Drive detection
from pathlib import Path
import pandas as pd
import numpy as np
import json, warnings
from datetime import datetime
warnings.filterwarnings("ignore")

NOTEBOOK_VERSION = "14_v13_sensitivity_use_saved_v7_predictions_bootstrap_fixed"
print("NOTEBOOK VERSION:", NOTEBOOK_VERSION)

candidates = [
    Path("/content/drive/MyDrive"),
    Path("/content/gdrive/MyDrive"),
    Path("/content/gdrive2/MyDrive"),
    Path("/content/gdrive_diag/MyDrive"),
    Path("/content/gdrive_diag2/MyDrive"),
]

ROOT = None
for c in candidates:
    if (c / "avian_influenza_project/processed/model_outputs_riskmap_eval").exists():
        ROOT = c
        print("Using existing Drive:", ROOT)
        break

if ROOT is None:
    from google.colab import drive
    mp = Path("/content/gdrive_v11")
    mp.mkdir(parents=True, exist_ok=True)
    drive.mount(str(mp), force_remount=False)
    ROOT = mp / "MyDrive"

PROC_DIR = ROOT / "avian_influenza_project/processed"
RESULT_DIR = PROC_DIR / "model_outputs_riskmap_eval"

PANEL_PATH = PROC_DIR / "hpai_weekly_grid_panel_with_previous_week_weather_model_ready.parquet"
EVENT_PATH = RESULT_DIR / "10_event_occurrence_type_classification.csv"
PRED_PATH = RESULT_DIR / "14_v7_all_grid_predictions_event_weeks.parquet"
SUMMARY_PATH = RESULT_DIR / "14_v7_summary_by_model_previous_week_weather.csv"

for p in [PANEL_PATH, EVENT_PATH, PRED_PATH, SUMMARY_PATH]:
    print(p.name, "exists =", p.exists())
    if not p.exists():
        raise FileNotFoundError(p)


In [ ]:

# 2. Sensitivity definitions
SENSITIVITY_DEFINITIONS = [
    {"definition": "4w_30km", "lookback_weeks": 4, "radius_km": 30.0, "primary": False},
    {"definition": "8w_20km", "lookback_weeks": 8, "radius_km": 20.0, "primary": False},
    {"definition": "8w_30km", "lookback_weeks": 8, "radius_km": 30.0, "primary": True},
    {"definition": "8w_50km", "lookback_weeks": 8, "radius_km": 50.0, "primary": False},
    {"definition": "12w_30km", "lookback_weeks": 12, "radius_km": 30.0, "primary": False},
]

TOPK_LIST = [0.01, 0.05, 0.10, 0.20, 0.30]
BOOTSTRAP_REPS = 20000
BOOTSTRAP_SEED = 20260617

BASELINE_MODEL = "baseline_extratrees_geo_season_previous_week_weather"
EXTERNAL_MODEL = "external_extratrees_geo_season_previous_week_weather_gis"

print(pd.DataFrame(SENSITIVITY_DEFINITIONS).to_string(index=False))


In [ ]:

# 3. Load source data and the FINAL saved 14_v7 predictions
panel = pd.read_parquet(PANEL_PATH)
events_original = pd.read_csv(EVENT_PATH)
pred = pd.read_parquet(PRED_PATH)
final_summary = pd.read_csv(SUMMARY_PATH)

panel["week_start"] = pd.to_datetime(panel["week_start"])
events_original["week_start"] = pd.to_datetime(events_original["week_start"])
pred["week_start"] = pd.to_datetime(pred["week_start"])

panel["outbreak_binary"] = pd.to_numeric(
    panel["outbreak_binary"], errors="coerce"
).fillna(0).astype(int)

evaluation_events = (
    events_original[["grid_id", "week_start"]]
    .drop_duplicates()
    .sort_values(["week_start", "grid_id"])
    .reset_index(drop=True)
)

print("Panel rows =", len(panel))
print("Full-panel outbreak-positive grid-weeks =", int(panel["outbreak_binary"].sum()))
print("Evaluation events =", len(evaluation_events))
print("Saved prediction rows =", len(pred))
print("Saved prediction event weeks =", pred["week_start"].nunique())
print("Saved prediction models =", pred["model_name"].unique().tolist())

if len(evaluation_events) != 61:
    raise AssertionError(f"Expected 61 evaluation events, got {len(evaluation_events)}")

if pred["week_start"].nunique() != 35:
    raise AssertionError(
        f"Expected 35 saved event weeks in final 14_v7 predictions, got {pred['week_start'].nunique()}"
    )

grid_counts = pred.groupby(["model_name", "week_start"])["grid_id"].nunique()
if not (grid_counts == 5491).all():
    print(grid_counts[grid_counts != 5491])
    raise AssertionError("Some saved final 14_v7 model-weeks do not contain all 5,491 grids.")

print("\nFinal 14_v7 summary:")
print(final_summary.to_string(index=False))


In [ ]:

# 4. Coordinates and prior outbreak-history table
grid_coords = (
    panel[["grid_id", "grid_lat", "grid_lon"]]
    .drop_duplicates("grid_id")
    .copy()
)

if len(grid_coords) != 5491:
    raise AssertionError(f"Expected 5,491 grid coordinates, got {len(grid_coords)}")

positive_history = (
    panel.loc[
        panel["outbreak_binary"].eq(1),
        ["grid_id", "week_start", "grid_lat", "grid_lon"]
    ]
    .drop_duplicates(["grid_id", "week_start"])
    .sort_values(["week_start", "grid_id"])
    .reset_index(drop=True)
)

event_coords = evaluation_events.merge(
    grid_coords, on="grid_id", how="left", validate="many_to_one"
)

if event_coords[["grid_lat", "grid_lon"]].isna().any().any():
    raise ValueError("Some evaluation events have missing grid coordinates.")

print("Positive-history rows =", len(positive_history))


In [ ]:

# 5. Recompute event classification for each sensitivity definition
EARTH_RADIUS_KM = 6371.0088

def haversine_km(lat1, lon1, lat2, lon2):
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    )
    return 2.0 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

def classify_events(events_df, history_df, lookback_weeks, radius_km):
    rows = []

    for ev in events_df.itertuples(index=False):
        event_week = pd.Timestamp(ev.week_start)
        lower = event_week - pd.Timedelta(weeks=int(lookback_weeks))

        # Only information strictly before the target week.
        hist = history_df[
            (history_df["week_start"] >= lower)
            & (history_df["week_start"] < event_week)
        ].copy()

        same_grid_prior = bool((hist["grid_id"] == ev.grid_id).any())

        neighbor_prior = False
        nearest_prior_km = np.nan

        if len(hist) > 0:
            d = haversine_km(
                float(ev.grid_lat),
                float(ev.grid_lon),
                hist["grid_lat"].to_numpy(float),
                hist["grid_lon"].to_numpy(float),
            )
            nearest_prior_km = float(np.min(d))

            other_grid = hist["grid_id"].to_numpy() != ev.grid_id
            if np.any(other_grid):
                neighbor_prior = bool(
                    np.any(d[other_grid] <= float(radius_km))
                )

        if same_grid_prior:
            occurrence_type = "same_grid_continuation"
        elif neighbor_prior:
            occurrence_type = "neighbor_continuation"
        else:
            occurrence_type = "first_occurrence_like"

        rows.append({
            "grid_id": ev.grid_id,
            "week_start": event_week,
            "lookback_weeks": int(lookback_weeks),
            "radius_km": float(radius_km),
            "same_grid_prior": same_grid_prior,
            "neighbor_prior": neighbor_prior,
            "nearest_prior_outbreak_km": nearest_prior_km,
            "occurrence_type": occurrence_type,
        })

    return pd.DataFrame(rows)

parts = []
for cfg in SENSITIVITY_DEFINITIONS:
    x = classify_events(
        event_coords,
        positive_history,
        cfg["lookback_weeks"],
        cfg["radius_km"]
    )
    x["definition"] = cfg["definition"]
    x["primary"] = cfg["primary"]
    parts.append(x)

sensitivity_events = pd.concat(parts, ignore_index=True)

classification_summary = (
    sensitivity_events
    .groupby(
        ["definition", "lookback_weeks", "radius_km", "primary", "occurrence_type"]
    )
    .size()
    .rename("events")
    .reset_index()
)

classification_pivot = classification_summary.pivot_table(
    index=["definition", "lookback_weeks", "radius_km", "primary"],
    columns="occurrence_type",
    values="events",
    fill_value=0,
    aggfunc="sum",
).reset_index()

for c in ["first_occurrence_like", "same_grid_continuation", "neighbor_continuation"]:
    if c not in classification_pivot.columns:
        classification_pivot[c] = 0

classification_pivot["total_events"] = (
    classification_pivot["first_occurrence_like"]
    + classification_pivot["same_grid_continuation"]
    + classification_pivot["neighbor_continuation"]
)

print(classification_pivot.to_string(index=False))


In [ ]:

# 6. Validate primary 8w/30km event classification
r = classification_pivot[
    classification_pivot["definition"].eq("8w_30km")
].iloc[0]

expected = {
    "total_events": 61,
    "first_occurrence_like": 49,
    "same_grid_continuation": 10,
    "neighbor_continuation": 2,
}

checks = []
for k, exp in expected.items():
    obs = int(r[k])
    checks.append({
        "check": k,
        "observed": obs,
        "expected": exp,
        "passed": obs == exp,
    })

classification_validation = pd.DataFrame(checks)
print(classification_validation.to_string(index=False))

if not classification_validation["passed"].all():
    raise AssertionError("Primary 8w/30km classification did not reproduce 49/10/2.")

print("PRIMARY CLASSIFICATION REPRODUCTION PASSED")


In [ ]:

# 7. Verify that all STRICT-event weeks required by the sensitivity definitions
#    are already covered by saved final 14_v7 predictions.
#
# Important:
# We do NOT need predictions for every one of the 61 evaluation events.
# We need predictions only for weeks containing events classified as
# first_occurrence_like under at least one sensitivity definition.

strict_event_weeks = set(
    pd.to_datetime(
        sensitivity_events.loc[
            sensitivity_events["occurrence_type"].eq("first_occurrence_like"),
            "week_start",
        ]
    ).unique()
)

saved_weeks = set(pd.to_datetime(pred["week_start"]).unique())
missing_strict_weeks = sorted(strict_event_weeks - saved_weeks)

print("Strict-event weeks required across all sensitivity definitions =", len(strict_event_weeks))
print("Saved final 14_v7 prediction weeks =", len(saved_weeks))
print("Missing required strict-event weeks =", len(missing_strict_weeks))

if missing_strict_weeks:
    print("Missing required weeks:", missing_strict_weeks)

    # Show exactly which definitions/events require those weeks.
    missing_detail = sensitivity_events[
        sensitivity_events["week_start"].isin(missing_strict_weeks)
        & sensitivity_events["occurrence_type"].eq("first_occurrence_like")
    ][
        ["definition", "lookback_weeks", "radius_km",
         "grid_id", "week_start", "occurrence_type"]
    ].sort_values(["week_start", "definition", "grid_id"])

    print("\nMissing-week strict events:")
    print(missing_detail.to_string(index=False))

    raise AssertionError(
        "Some weeks containing strict first-occurrence-like events under the "
        "sensitivity definitions are not present in the saved final 14_v7 predictions. "
        "Only those missing weeks require additional exact-pipeline scoring."
    )

print("ALL REQUIRED STRICT-EVENT WEEKS ARE COVERED BY SAVED FINAL 14_v7 PREDICTIONS")


In [ ]:

# 8. Attach saved final predictions to sensitivity-defined strict events
case_parts = []

for cfg in SENSITIVITY_DEFINITIONS:
    d = cfg["definition"]

    ev = sensitivity_events[
        sensitivity_events["definition"].eq(d)
        & sensitivity_events["occurrence_type"].eq("first_occurrence_like")
    ][
        ["grid_id", "week_start", "definition", "lookback_weeks", "radius_km"]
    ].copy()

    for model_name in [BASELINE_MODEL, EXTERNAL_MODEL]:
        pp = pred[pred["model_name"].eq(model_name)].copy()

        m = ev.merge(
            pp,
            on=["grid_id", "week_start"],
            how="left",
            validate="one_to_one",
        )

        if "model_name" not in m.columns:
            m["model_name"] = model_name

        if m["risk_rank"].isna().any():
            print(m[m["risk_rank"].isna()])
            raise AssertionError(f"Unmatched saved predictions for {d}, {model_name}")

        # Use final saved risk_rank/risk_percentile directly.
        for k in TOPK_LIST:
            m[f"top{int(k*100)}_hit"] = (
                m["risk_rank"] <= np.ceil(5491 * k)
            )

        case_parts.append(m)

sensitivity_case_results = pd.concat(case_parts, ignore_index=True)

print(
    sensitivity_case_results
    .groupby(["definition", "model_name"])
    .size()
    .rename("event_rows")
    .reset_index()
    .to_string(index=False)
)


In [ ]:

# 9. Critical validation: primary 8w/30km MUST equal final saved 14_v7 summary exactly
def summarize_model(g):
    return {
        "events": len(g),
        "top1_events": int(g["top1_hit"].sum()),
        "top5_events": int(g["top5_hit"].sum()),
        "top10_events": int(g["top10_hit"].sum()),
        "top20_events": int(g["top20_hit"].sum()),
        "top30_events": int(g["top30_hit"].sum()),
        "mean_risk_percentile": float(g["risk_percentile"].mean()),
        "median_risk_percentile": float(g["risk_percentile"].median()),
    }

validation_rows = []

for model_name in [BASELINE_MODEL, EXTERNAL_MODEL]:
    g = sensitivity_case_results[
        sensitivity_case_results["definition"].eq("8w_30km")
        & sensitivity_case_results["model_name"].eq(model_name)
    ].copy()

    obs = summarize_model(g)
    ref = final_summary[final_summary["model_name"].eq(model_name)].iloc[0]

    comparisons = {
        "events": (obs["events"], int(ref["events"])),
        "top1_events": (obs["top1_events"], int(ref["top1_events"])),
        "top5_events": (obs["top5_events"], int(ref["top5_events"])),
        "top10_events": (obs["top10_events"], int(ref["top10_events"])),
        "top20_events": (obs["top20_events"], int(ref["top20_events"])),
        "top30_events": (obs["top30_events"], int(ref["top30_events"])),
    }

    for metric, (a, b) in comparisons.items():
        validation_rows.append({
            "model_name": model_name,
            "metric": metric,
            "observed": a,
            "reference": b,
            "passed": a == b,
        })

    for metric in ["mean_risk_percentile", "median_risk_percentile"]:
        a = obs[metric]
        b = float(ref[metric])
        validation_rows.append({
            "model_name": model_name,
            "metric": metric,
            "observed": a,
            "reference": b,
            "passed": bool(np.isclose(a, b, atol=1e-12, rtol=0)),
        })

primary_prediction_validation = pd.DataFrame(validation_rows)
print(primary_prediction_validation.to_string(index=False))

if not primary_prediction_validation["passed"].all():
    raise AssertionError(
        "Joining sensitivity events to saved final 14_v7 predictions did not reproduce final results."
    )

print("FINAL 14_v7 PRIMARY RESULTS REPRODUCED EXACTLY FROM SAVED PREDICTIONS")


In [ ]:

# 10. Performance summary for all sensitivity definitions
rows = []

for (definition, model_name), g in sensitivity_case_results.groupby(
    ["definition", "model_name"]
):
    row = {
        "definition": definition,
        "model_name": model_name,
        "events": int(len(g)),
        "mean_risk_percentile": float(g["risk_percentile"].mean()),
        "median_risk_percentile": float(g["risk_percentile"].median()),
        "mean_rank": float(g["risk_rank"].mean()),
        "median_rank": float(g["risk_rank"].median()),
    }

    for k in TOPK_LIST:
        c = f"top{int(k*100)}_hit"
        row[f"top{int(k*100)}_events"] = int(g[c].sum())
        row[f"top{int(k*100)}_capture_rate"] = float(g[c].mean())

    rows.append(row)

performance_by_definition_model = pd.DataFrame(rows).merge(
    pd.DataFrame(SENSITIVITY_DEFINITIONS),
    on="definition",
    how="left",
    validate="many_to_one",
)

print(performance_by_definition_model.to_string(index=False))


In [ ]:

# 11. Paired baseline-vs-GIS sensitivity comparison
from scipy.stats import binomtest

comparison_rows = []
paired_parts = []

for cfg in SENSITIVITY_DEFINITIONS:
    d = cfg["definition"]

    # Reset bootstrap RNG for each definition so identical event sets
    # produce identical bootstrap CIs and the primary 8w/30km CI
    # reproduces the final 14_v7 bootstrap result.
    rng = np.random.default_rng(BOOTSTRAP_SEED)

    b = sensitivity_case_results[
        sensitivity_case_results["definition"].eq(d)
        & sensitivity_case_results["model_name"].eq(BASELINE_MODEL)
    ].copy()

    e = sensitivity_case_results[
        sensitivity_case_results["definition"].eq(d)
        & sensitivity_case_results["model_name"].eq(EXTERNAL_MODEL)
    ].copy()

    keep = [
        "grid_id", "week_start", "risk_rank", "risk_percentile",
        "top10_hit", "top20_hit"
    ]

    p = b[keep].merge(
        e[keep],
        on=["grid_id", "week_start"],
        suffixes=("_baseline", "_external"),
        validate="one_to_one",
    )

    p["definition"] = d
    p["delta_percentile"] = (
        p["risk_percentile_external"]
        - p["risk_percentile_baseline"]
    )

    p["gain_to_top10"] = (
        (~p["top10_hit_baseline"]) & p["top10_hit_external"]
    )
    p["loss_from_top10"] = (
        p["top10_hit_baseline"] & (~p["top10_hit_external"])
    )
    p["stable_top10"] = (
        p["top10_hit_baseline"] & p["top10_hit_external"]
    )
    p["stable_miss"] = (
        (~p["top10_hit_baseline"]) & (~p["top10_hit_external"])
    )

    paired_parts.append(p)

    n = len(p)
    base10 = float(p["top10_hit_baseline"].mean())
    ext10 = float(p["top10_hit_external"].mean())
    base20 = float(p["top20_hit_baseline"].mean())
    ext20 = float(p["top20_hit_external"].mean())

    # paired event bootstrap
    idx = np.arange(n)
    boot = np.empty(BOOTSTRAP_REPS, dtype=float)

    for r in range(BOOTSTRAP_REPS):
        s = rng.choice(idx, size=n, replace=True)
        boot[r] = (
            p.iloc[s]["top10_hit_external"].mean()
            - p.iloc[s]["top10_hit_baseline"].mean()
        )

    ci_low, ci_high = np.quantile(boot, [0.025, 0.975])

    gain = int(p["gain_to_top10"].sum())
    loss = int(p["loss_from_top10"].sum())
    discordant = gain + loss

    exact_p = (
        float(
            binomtest(
                gain,
                n=discordant,
                p=0.5,
                alternative="two-sided",
            ).pvalue
        )
        if discordant > 0
        else 1.0
    )

    comparison_rows.append({
        "definition": d,
        "lookback_weeks": cfg["lookback_weeks"],
        "radius_km": cfg["radius_km"],
        "primary": cfg["primary"],
        "strict_events": n,
        "baseline_top10_rate": base10,
        "external_top10_rate": ext10,
        "top10_difference_pp": 100 * (ext10 - base10),
        "top10_bootstrap_ci_low_pp": 100 * float(ci_low),
        "top10_bootstrap_ci_high_pp": 100 * float(ci_high),
        "baseline_top20_rate": base20,
        "external_top20_rate": ext20,
        "top20_difference_pp": 100 * (ext20 - base20),
        "mean_delta_percentile": float(p["delta_percentile"].mean()),
        "median_delta_percentile": float(p["delta_percentile"].median()),
        "gain_to_top10": gain,
        "loss_from_top10": loss,
        "stable_top10": int(p["stable_top10"].sum()),
        "stable_miss": int(p["stable_miss"].sum()),
        "discordant_top10": discordant,
        "exact_binomial_p": exact_p,
    })

sensitivity_comparison = pd.DataFrame(comparison_rows)
sensitivity_paired_cases = pd.concat(paired_parts, ignore_index=True)

print("\n===== FINAL SENSITIVITY COMPARISON =====")
print(sensitivity_comparison.to_string(index=False))


In [ ]:

# 12. Build manuscript-ready Supplementary Table S5
table_s5 = sensitivity_comparison[[
    "definition",
    "lookback_weeks",
    "radius_km",
    "primary",
    "strict_events",
    "baseline_top10_rate",
    "external_top10_rate",
    "top10_difference_pp",
    "top10_bootstrap_ci_low_pp",
    "top10_bootstrap_ci_high_pp",
    "baseline_top20_rate",
    "external_top20_rate",
    "top20_difference_pp",
    "mean_delta_percentile",
    "median_delta_percentile",
    "gain_to_top10",
    "loss_from_top10",
    "exact_binomial_p",
]].copy()

order = {x["definition"]: i for i, x in enumerate(SENSITIVITY_DEFINITIONS)}
table_s5["_order"] = table_s5["definition"].map(order)
table_s5 = (
    table_s5.sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)

print("\n===== SUPPLEMENTARY TABLE S5 =====")
print(table_s5.to_string(index=False))


In [ ]:

# 13. Save outputs
outputs = {
    "14_v13_sensitivity_event_classification_all_definitions.csv": sensitivity_events,
    "14_v13_sensitivity_event_classification_summary.csv": classification_pivot,
    "14_v13_primary_classification_validation.csv": classification_validation,
    "14_v13_primary_saved_prediction_validation.csv": primary_prediction_validation,
    "14_v13_sensitivity_case_results_all_definitions.csv": sensitivity_case_results,
    "14_v13_sensitivity_performance_by_definition_model.csv": performance_by_definition_model,
    "14_v13_sensitivity_paired_baseline_vs_gis_cases.csv": sensitivity_paired_cases,
    "14_v13_sensitivity_baseline_vs_gis_summary.csv": sensitivity_comparison,
    "14_v13_supplementary_table_S5_first_occurrence_sensitivity.csv": table_s5,
}

for name, df in outputs.items():
    p = RESULT_DIR / name
    df.to_csv(p, index=False, encoding="utf-8-sig")
    print("saved:", p)

run_summary = {
    "notebook_version": NOTEBOOK_VERSION,
    "run_timestamp": datetime.now().isoformat(),
    "analysis_strategy": "reuse_saved_final_14_v7_predictions_no_refitting",
    "saved_prediction_path": str(PRED_PATH),
    "sensitivity_definitions": SENSITIVITY_DEFINITIONS,
    "primary_classification_validation_passed": bool(
        classification_validation["passed"].all()
    ),
    "primary_saved_prediction_validation_passed": bool(
        primary_prediction_validation["passed"].all()
    ),
}

p = RESULT_DIR / "14_v13_sensitivity_run_summary.json"
p.write_text(
    json.dumps(run_summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("saved:", p)



## What to send back

After all cells finish successfully, copy and paste the printed section:

`===== SUPPLEMENTARY TABLE S5 =====`

If the notebook stops at the "all evaluation event weeks are covered" check, paste that output instead.
